# DB7 Exercise B: three-branch model versus all-sensor baseline

One independently trained B0 baseline and C1 three-branch model per subject. 22 subjects; 400 ms windows; 100 ms stride; fixed 4/1/1 repetitions. GitHub runs seed bases 42, 43 and 44 in separate Kaggle jobs. No RMS add-on, T-EKIM, extra centering, augmentation or LDA gate is used in this first comparison. The historical test split is exploratory because it has already informed model development. No packages are installed in this notebook. Enable the T4 GPU and attach rayaanraza1/ninapro-db7.


## 1. Existing preprocessing, baseline and trainer
The definitions below preserve the previous protocol. Input means/SDs are fitted only on training windows.


In [ ]:
import os, gc, time, json, warnings, random, re, shutil
from pathlib import Path
from copy import deepcopy
from math import gcd
import numpy as np
import pandas as pd
from scipy import io
from scipy.signal import resample_poly, butter, sosfiltfilt, iirnotch, filtfilt
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, roc_curve
import matplotlib
import hashlib
import torch
import torch.nn as nn
import torch.nn.functional as F
import sys, platform, traceback, zipfile
from datetime import datetime, timezone
from sklearn.metrics import classification_report, balanced_accuracy_score
from scipy.signal import welch
matplotlib.use('Agg')


In [ ]:
def _find_kaggle_input() -> Path:
    base = Path('/kaggle/input')
    if not base.exists():
        return Path('/kaggle/input/ninapro-db7/Dataset')

    def _has_subjects(p):
        return p.is_dir() and any((c.is_dir() and c.name.lower().startswith('subject_') for c in p.iterdir()))

    def _search(root, depth=0):
        if depth > 5:
            return None
        if _has_subjects(root):
            return root
        try:
            for child in sorted(root.iterdir()):
                if child.is_dir():
                    result = _search(child, depth + 1)
                    if result is not None:
                        return result
        except PermissionError:
            pass
        return None
    result = _search(base)
    return result if result else Path('/kaggle/input/ninapro-db7/Dataset')


In [ ]:
class Config:
    KAGGLE_INPUT = _find_kaggle_input()
    KAGGLE_WORKING = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd() / 'eda_working'
    EXERCISE_IDS = (1,)
    GESTURE_MIN, GESTURE_MAX, N_CLASSES = (1, 17, 17)
    SUBJECTS = list(range(1, 23))
    RUN_SUBJECTS = SUBJECTS.copy()
    INTACT_SUBJECTS = list(range(1, 21))
    AMPUTEE_SUBJECTS = [21, 22]
    SEED = 42
    MODEL_SEED = 42
    EMG_FS, ACC_FS, TARGET_FS = (2000, 128, 2000)
    EMG_KEY, ACC_KEY, LBL_KEY = ('emg', 'acc', 'restimulus')
    N_EMG_CH = 12
    USE_ACC = True
    REPS_PER_GESTURE, TRAIN_REPS, VAL_REPS, TEST_REPS = (6, 4, 1, 1)
    BANDPASS_LOW_HZ, BANDPASS_HIGH_HZ, FILTER_ORDER = (20.0, 450.0, 4)
    NOTCH_HZ, NOTCH_Q = (50.0, 30.0)
    WIN_MS, STEP_MS, TRIM_MS = (400, 100, 100)
    WIN_SAMPLES, STEP_SAMPLES, TRIM_SAMPLES = (800, 200, 200)
    DROPOUT, LR, WEIGHT_DECAY, GRAD_CLIP = (0.15, 0.0003, 0.0001, 5.0)
    MIN_EPOCHS, MAX_EPOCHS, PATIENCE, BATCH_SIZE = (20, 150, 15, 128)
    NUM_WORKERS = 0
    USE_ADAMW, AUGMENT_TRAIN, EVALUATE_TEST, REFIT_ON_TRAIN_PLUS_VAL = (False, False, False, False)
    LABEL_SMOOTHING = 0.0
    EMG_GAIN_STD, EMG_NOISE_STD = (0.1, 0.01)
    RUN_CNN = True
    RAW_CASES_PER_SUBJECT = 2
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    AUTOMATION = {}


In [ ]:
class RepetitionEMGFilter:
    """Zero-phase EMG filtering applied to one already-assigned repetition only."""

    def __init__(self):
        nyquist = Config.EMG_FS / 2.0
        self.sos = butter(Config.FILTER_ORDER, [Config.BANDPASS_LOW_HZ / nyquist, Config.BANDPASS_HIGH_HZ / nyquist], btype='bandpass', output='sos')
        self.b_notch, self.a_notch = iirnotch(Config.NOTCH_HZ / nyquist, Config.NOTCH_Q)

    def apply(self, repetition_emg):
        repetition_emg = np.asarray(repetition_emg, dtype=np.float32)
        if repetition_emg.ndim != 2:
            raise ValueError(f'Expected (time,channels), got {repetition_emg.shape}.')
        if len(repetition_emg) < 64:
            raise RuntimeError(f'EMG repetition unexpectedly short: {len(repetition_emg)} samples.')
        filtered = sosfiltfilt(self.sos, repetition_emg, axis=0)
        filtered = filtfilt(self.b_notch, self.a_notch, filtered, axis=0)
        return filtered.astype(np.float32)


In [ ]:
class SubjectLoader:

    @staticmethod
    def _split_repetition_indices(sid, gesture):
        rng = np.random.default_rng(Config.SEED + 1009 * int(sid) + 9176 * int(gesture))
        perm = rng.permutation(Config.REPS_PER_GESTURE).tolist()
        test_idx = sorted(perm[:Config.TEST_REPS])
        val_idx = sorted(perm[Config.TEST_REPS:Config.TEST_REPS + Config.VAL_REPS])
        train_idx = sorted(perm[Config.TEST_REPS + Config.VAL_REPS:])
        if len(train_idx) != Config.TRAIN_REPS or len(val_idx) != Config.VAL_REPS or len(test_idx) != Config.TEST_REPS:
            raise RuntimeError('Unexpected repetition split size.')
        return (train_idx, val_idx, test_idx)


In [ ]:
class ParallelMultiKernelBlock(nn.Module):
    """
    True multi-kernel block: parallel branches with different kernel sizes
    (default 3, 5, 7) processed at the SAME depth and concatenated along the
    channel dimension, then merged with a 1x1 conv. This captures multi-scale
    temporal patterns simultaneously (unlike a sequential 7->5->3 design,
    which only changes kernel size across depth, not within one stage).
    """

    def __init__(self, in_ch, out_ch, kernels=(3, 5, 7), pool=True, dropout=0.1):
        super().__init__()
        for k in kernels:
            assert k % 2 == 1, f"kernel size {k} must be odd so that padding=kernel//2 gives symmetric 'same' padding"
        branch_sizes = self._split_channels(out_ch, len(kernels))
        self.branches = nn.ModuleList([nn.Sequential(nn.Conv1d(in_ch, b_ch, k, padding=k // 2, bias=False), nn.BatchNorm1d(b_ch), nn.ReLU(inplace=True), nn.Conv1d(b_ch, b_ch, k, padding=k // 2, bias=False), nn.BatchNorm1d(b_ch), nn.ReLU(inplace=True)) for k, b_ch in zip(kernels, branch_sizes)])
        self.merge = nn.Sequential(nn.Conv1d(out_ch, out_ch, 1, bias=False), nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True))
        self.drop = nn.Dropout1d(dropout) if dropout > 0 else nn.Identity()
        self.pool = nn.MaxPool1d(2) if pool else nn.Identity()

    @staticmethod
    def _split_channels(total, n):
        base, rem = divmod(total, n)
        return [base + 1 if i < rem else base for i in range(n)]

    def forward(self, x):
        x = torch.cat([branch(x) for branch in self.branches], dim=1)
        x = self.merge(x)
        x = self.drop(x)
        return self.pool(x)


In [ ]:
class ChannelAttentionBlock(nn.Module):
    """
    Squeeze-and-Excitation style CHANNEL attention. Learns which learned feature channels
    matter most; it does NOT attend across time
    steps. Named explicitly so it isn't confused with temporal attention.
    """

    def __init__(self, in_ch, reduction=4):
        super().__init__()
        reduced = max(in_ch // reduction, 1)
        self.attention = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Conv1d(in_ch, reduced, 1), nn.ReLU(inplace=True), nn.Conv1d(reduced, in_ch, 1), nn.Sigmoid())

    def forward(self, x):
        return x * self.attention(x)


In [ ]:
class TemporalAttentionPool(nn.Module):
    """
    Learned attention-weighted pooling over the time axis, replacing plain
    global average pooling. A 1x1 conv scores every time step, softmax turns
    scores into weights, and the weighted sum replaces a uniform mean — so
    the model can down-weight uninformative (e.g. resting) time steps instead
    of averaging them in blindly.
    """

    def __init__(self, in_ch):
        super().__init__()
        self.score = nn.Conv1d(in_ch, 1, kernel_size=1)

    def forward(self, x):
        weights = torch.softmax(self.score(x), dim=-1)
        return (x * weights).sum(dim=-1)


In [ ]:
class OriginalMultiKernelAttention1DCNN(nn.Module):

    def __init__(self, n_channels, n_classes, dropout=0.15):
        super().__init__()
        self.n_channels = int(n_channels)
        self.stage1 = ParallelMultiKernelBlock(n_channels, 64, kernels=(3, 5, 7), pool=True, dropout=dropout)
        self.attn1 = ChannelAttentionBlock(64)
        self.stage2 = ParallelMultiKernelBlock(64, 128, kernels=(3, 5, 7), pool=True, dropout=dropout)
        self.attn2 = ChannelAttentionBlock(128)
        self.stage3 = ParallelMultiKernelBlock(128, 256, kernels=(3, 5, 7), pool=False, dropout=dropout)
        self.attn3 = ChannelAttentionBlock(256)
        self.temporal_pool = TemporalAttentionPool(256)
        self.head = nn.Sequential(nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Dropout(dropout), nn.Linear(128, n_classes))

    def forward(self, x):
        x = self.stage1(x)
        x = self.attn1(x)
        x = self.stage2(x)
        x = self.attn2(x)
        x = self.stage3(x)
        x = self.attn3(x)
        x = self.temporal_pool(x)
        return self.head(x)

    def count_params(self):
        return sum((parameter.numel() for parameter in self.parameters() if parameter.requires_grad))


In [ ]:
class Trainer:

    def __init__(self, model, save_path: Path, n_classes: int):
        self.model = model.to(Config.DEVICE)
        self.save_path = save_path
        self.n_classes = n_classes
        self.history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
        self.best_epoch = 0
        self.best_val_loss = float('inf')
        self.train_wall = 0.0

    def _run_epoch(self, loader, optimizer=None, criterion=None):
        training = optimizer is not None
        self.model.train(training)
        total_loss, correct, total = (0.0, 0, 0)
        context = torch.enable_grad() if training else torch.no_grad()
        with context:
            for X, y in loader:
                X = X.to(Config.DEVICE, non_blocking=True)
                y = y.to(Config.DEVICE, non_blocking=True)
                if training and Config.AUGMENT_TRAIN:
                    X = X.clone()
                    emg = X[:, :Config.N_EMG_CH]
                    gain = torch.exp(Config.EMG_GAIN_STD * torch.randn(emg.shape[0], emg.shape[1], 1, device=emg.device))
                    X[:, :Config.N_EMG_CH] = emg * gain + Config.EMG_NOISE_STD * torch.randn_like(emg)
                logits = self.model(X)
                loss = criterion(logits, y)
                if training:
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), Config.GRAD_CLIP)
                    optimizer.step()
                total_loss += loss.item() * len(y)
                correct += (logits.argmax(1) == y).sum().item()
                total += len(y)
        return (total_loss / max(total, 1), correct / max(total, 1))

    def fit(self, train_loader, val_loader):
        optimizer = (torch.optim.AdamW if Config.USE_ADAMW else Adam)(self.model.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)
        criterion = nn.CrossEntropyLoss(label_smoothing=Config.LABEL_SMOOTHING)
        scheduler = CosineAnnealingLR(optimizer, T_max=Config.MAX_EPOCHS)
        patience_count = 0
        start_wall = time.perf_counter()
        for epoch in range(1, Config.MAX_EPOCHS + 1):
            tr_loss, tr_acc = self._run_epoch(train_loader, optimizer, criterion)
            vl_loss, vl_acc = self._run_epoch(val_loader, criterion=criterion)
            scheduler.step()
            self.history['train_loss'].append(tr_loss)
            self.history['val_loss'].append(vl_loss)
            self.history['train_acc'].append(tr_acc)
            self.history['val_acc'].append(vl_acc)
            if vl_loss < self.best_val_loss:
                self.best_val_loss = float(vl_loss)
                self.best_epoch = int(epoch)
                patience_count = 0
                torch.save(self.model.state_dict(), self.save_path)
            else:
                patience_count += 1
            if epoch == 1 or epoch % 10 == 0:
                print(f'  Epoch {epoch:3d} | train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | val_loss={vl_loss:.4f} val_acc={vl_acc:.4f}')
            if epoch >= Config.MIN_EPOCHS and patience_count >= Config.PATIENCE:
                print(f'  Early stop at epoch {epoch} (patience={Config.PATIENCE})')
                break
        self.train_wall = time.perf_counter() - start_wall
        print(f'  Selection training: {self.train_wall:.1f} s | best epoch={self.best_epoch} | best val_loss={self.best_val_loss:.4f}')
        return self

    def fit_fixed_epochs(self, train_loader, epochs: int):
        """Fresh final model training on train+validation after epoch selection."""
        optimizer = (torch.optim.AdamW if Config.USE_ADAMW else Adam)(self.model.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)
        criterion = nn.CrossEntropyLoss(label_smoothing=Config.LABEL_SMOOTHING)
        scheduler = CosineAnnealingLR(optimizer, T_max=Config.MAX_EPOCHS)
        start_wall = time.perf_counter()
        for epoch in range(1, int(epochs) + 1):
            loss, acc = self._run_epoch(train_loader, optimizer, criterion)
            scheduler.step()
            if epoch == 1 or epoch % 10 == 0 or epoch == int(epochs):
                print(f'  Refit epoch {epoch:3d}/{int(epochs)} | loss={loss:.4f} | acc={acc:.4f}')
        self.train_wall = time.perf_counter() - start_wall
        torch.save(self.model.state_dict(), self.save_path)
        print(f'  Refit training: {self.train_wall:.1f} s')
        return self


In [ ]:
def json_write(path, obj):

    def convert(x):
        if isinstance(x, np.ndarray):
            return x.tolist()
        if isinstance(x, np.generic):
            return x.item()
        return str(x)
    Path(path).write_text(json.dumps(obj, indent=2, default=convert), encoding='utf-8')


In [ ]:
def save_figure(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=130, bbox_inches='tight')
    plt.close(fig)


In [ ]:
def probability_metrics(y, p):
    """Uncalibrated confidence diagnostics; multiclass Brier is a sum over classes."""
    pred = p.argmax(1)
    confidence = p.max(1)
    correct = pred == y
    bins = np.minimum((confidence * 10).astype(int), 9)
    calibration = []
    ece = 0.0
    for b in range(10):
        mask = bins == b
        n = int(mask.sum())
        acc = float(correct[mask].mean()) if n else None
        conf = float(confidence[mask].mean()) if n else None
        calibration.append(dict(bin=b, lower=b / 10, upper=(b + 1) / 10, count=n, accuracy=acc, confidence=conf))
        if n:
            ece += n / len(y) * abs(acc - conf)
    targets = np.eye(p.shape[1])[y]
    result = dict(accuracy=float(correct.mean()), balanced_accuracy=float(balanced_accuracy_score(y, pred)), f1_macro=float(f1_score(y, pred, labels=np.arange(p.shape[1]), average='macro', zero_division=0)), f1_weighted=float(f1_score(y, pred, average='weighted', zero_division=0)), nll=float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)).mean()), brier_multiclass=float(((p - targets) ** 2).sum(1).mean()), ece_10_bins=float(ece), high_confidence_error_fraction=float(((confidence >= 0.9) & ~correct).mean()), n_windows=int(len(y)))
    return (result, calibration)


In [ ]:
class DiagnosticTrainer(Trainer):
    """Preserves Trainer.fit selection logic; records each epoch without extra forwards."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.epoch_rows = []
        self.last_training = {}

    def _run_epoch(self, loader, optimizer=None, criterion=None):
        training = optimizer is not None
        self.model.train(training)
        total_loss, correct, total = (0.0, 0, 0)
        norms, pred_all, y_all = ([], [], [])
        tick = time.perf_counter()
        with torch.set_grad_enabled(training):
            for X, y in loader:
                X, y = (X.to(Config.DEVICE), y.to(Config.DEVICE))
                if training and Config.AUGMENT_TRAIN:
                    X = X.clone()
                    emg = X[:, :Config.N_EMG_CH]
                    gain = torch.exp(Config.EMG_GAIN_STD * torch.randn(emg.shape[0], emg.shape[1], 1, device=emg.device))
                    X[:, :Config.N_EMG_CH] = emg * gain + Config.EMG_NOISE_STD * torch.randn_like(emg)
                logits = self.model(X)
                loss = criterion(logits, y)
                if not torch.isfinite(loss):
                    raise FloatingPointError('Non-finite loss; inspect signal_health.csv')
                if training:
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    norm = torch.nn.utils.clip_grad_norm_(self.model.parameters(), Config.GRAD_CLIP)
                    if not torch.isfinite(norm):
                        raise FloatingPointError('Non-finite gradient norm')
                    norms.append(float(norm.item()))
                    optimizer.step()
                pred = logits.argmax(1)
                total_loss += loss.item() * len(y)
                correct += (pred == y).sum().item()
                total += len(y)
                pred_all.extend(pred.detach().cpu().tolist())
                y_all.extend(y.cpu().tolist())
        values = dict(loss=total_loss / total, accuracy=correct / total, f1_macro=float(f1_score(y_all, pred_all, labels=range(self.n_classes), average='macro', zero_division=0)), seconds=time.perf_counter() - tick)
        labels = np.asarray(y_all)
        guesses = np.asarray(pred_all)
        if not hasattr(self, 'class_epoch_rows'):
            self.class_epoch_rows = []
        for c in range(self.n_classes):
            selected = labels == c
            self.class_epoch_rows.append(dict(epoch=len(self.epoch_rows) + 1, split='train_online_dropout' if training else 'validation', gesture=c + Config.GESTURE_MIN, windows=int(selected.sum()), recall=float((guesses[selected] == c).mean())))
        pd.DataFrame(self.class_epoch_rows).to_csv(self.save_path.parent / 'class_learning_history.csv', index=False)
        if training:
            self.last_training = {'train_' + k: v for k, v in values.items()}
            self.last_training.update(lr=optimizer.param_groups[0]['lr'], gradient_norm_mean=float(np.mean(norms)), gradient_norm_max=float(np.max(norms)), gradient_clipped_fraction=float(np.mean(np.array(norms) > Config.GRAD_CLIP)))
        else:
            row = dict(epoch=len(self.epoch_rows) + 1, **self.last_training, **{'val_' + k: v for k, v in values.items()})
            self.epoch_rows.append(row)
            pd.DataFrame(self.epoch_rows).to_csv(self.save_path.parent / 'history.csv', index=False)
        return (values['loss'], values['accuracy'])


In [ ]:
def learning_report(trainer, directory):
    h = pd.DataFrame(trainer.epoch_rows)
    h.to_csv(directory / 'history.csv', index=False)
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for split in ('train', 'val'):
        axes[0, 0].plot(h.epoch, h[split + '_loss'], label=split)
        axes[0, 1].plot(h.epoch, h[split + '_accuracy'], label=split)
    axes[0, 0].set_ylabel('Cross entropy')
    axes[0, 1].set_ylabel('Accuracy')
    axes[1, 0].plot(h.epoch, h.lr)
    axes[1, 0].set_ylabel('Learning rate')
    axes[1, 1].plot(h.epoch, h.gradient_norm_mean, label='Mean before clipping')
    axes[1, 1].plot(h.epoch, h.gradient_norm_max, label='Max before clipping')
    axes[1, 1].axhline(Config.GRAD_CLIP, color='gray', linestyle='--')
    axes[1, 1].set_ylabel('Gradient norm')
    for ax in axes.flat:
        ax.axvline(trainer.best_epoch, color='red', linestyle=':', label='Selected epoch')
        ax.set_xlabel('Epoch')
        ax.legend(fontsize=8)
    fig.suptitle('Online training uses dropout; checkpoint gap is measured separately in eval mode')
    save_figure(fig, directory / 'learning_curves.png')


In [ ]:
def predict_with_diagnostics(model, dataset, retain_attention=False):
    """Evaluate the checkpoint; collect embeddings and optionally attention on every window."""
    model.eval()
    captured, probabilities, embeddings, attention = ({}, [], [], {})

    def capture(name):

        def hook(module, args, result):
            captured[name] = result.detach()
        return hook
    handles = [model.temporal_pool.register_forward_hook(capture('embedding'))]
    if retain_attention:
        handles += [model.temporal_pool.score.register_forward_hook(capture('time_scores'))]
        for stage in [1, 2, 3]:
            handles.append(getattr(model, f'attn{stage}').attention.register_forward_hook(capture(f'channel_stage{stage}')))
    try:
        with torch.no_grad():
            for signal, _ in DataLoader(dataset, batch_size=128, shuffle=False):
                logits = model(signal.to(Config.DEVICE))
                probabilities.append(logits.softmax(1).cpu().numpy())
                embeddings.append(captured['embedding'].cpu().numpy())
                if retain_attention:
                    for name in ['time_scores', 'channel_stage1', 'channel_stage2', 'channel_stage3']:
                        values = captured[name].softmax(-1) if name == 'time_scores' else captured[name]
                        attention.setdefault(name, []).append(values.squeeze(1 if name == 'time_scores' else -1).cpu().numpy())
    finally:
        for handle in handles:
            handle.remove()
    return (np.concatenate(probabilities), np.concatenate(embeddings), {k: np.concatenate(v) for k, v in attention.items()})


In [ ]:
CHANNEL_COUNTS = {'emg': 12, 'acc': 36, 'gyro': 36, 'mag': 36}


In [ ]:
def load_modalities(subject, directory):
    """Read only selected sensor arrays; use refined labels for every arm."""
    matches = sorted(Config.KAGGLE_INPUT.rglob(f'S{subject}_E1_A1.mat'))
    if len(matches) != 1:
        raise ValueError(f'Expected one S{subject}_E1_A1.mat under {Config.KAGGLE_INPUT}; found {len(matches)}')
    path = matches[0]
    keys = list(Config.MODALITIES) + ['restimulus', 'rerepetition', 'stimulus', 'subject', 'exercise']
    data = io.loadmat(path, variable_names=keys)
    labels = np.asarray(data['restimulus']).reshape(-1)
    repetitions = np.asarray(data['rerepetition']).reshape(-1)
    stimulus = np.asarray(data['stimulus']).reshape(-1)
    assert len(labels) == len(repetitions) == len(stimulus)
    assert set(np.unique(labels)) == set(range(18))
    assert int(data['exercise'].item()) == 1
    arrays, names, hashes = [], [], {}
    for modality in Config.MODALITIES:
        values = np.asarray(data[modality], dtype=np.float32)
        assert values.shape == (len(labels), CHANNEL_COUNTS[modality]), (modality, values.shape)
        assert np.isfinite(values).all(), f'Nonfinite {modality}: subject {subject}'
        hashes[modality] = hashlib.sha256(values.tobytes()).hexdigest()
        arrays.append(values)
        names.extend(f'{modality}_{i+1:02}' for i in range(values.shape[1]))
    signal = np.concatenate(arrays, axis=1)
    boundaries = np.r_[0, np.flatnonzero(np.diff(labels)) + 1, len(labels)]
    rows = []
    emg_filter = RepetitionEMGFilter() if 'emg' in Config.MODALITIES else None
    for gesture in range(1,18):
        runs = [(int(a),int(b)) for a,b in zip(boundaries[:-1],boundaries[1:]) if labels[a] == gesture]
        assert len(runs) == 6, (subject, gesture, len(runs))
        train_ids, val_ids, test_ids = SubjectLoader._split_repetition_indices(subject, gesture)
        seen = set()
        for index,(start,end) in enumerate(runs):
            native = np.unique(repetitions[start:end])
            assert len(native) == 1 and 1 <= native[0] <= 6 and int(native[0]) not in seen
            seen.add(int(native[0]))
            split = 'train' if index in train_ids else 'validation' if index in val_ids else 'test'
            if emg_filter is not None:
                # EMG is first in every combination containing it.
                signal[start:end,:12] = emg_filter.apply(signal[start:end,:12])
            rows.append(dict(subject=subject, gesture=gesture, native_repetition=int(native[0]),
                split=split,file_name=path.name,run_start=start,run_end=end))
    metadata = pd.DataFrame(rows)
    metadata.to_csv(directory/'repetition_inventory.csv',index=False)
    identity = dict(folder_subject=subject, internal_subject=int(data['subject'].item()),
        exercise=1,modalities=Config.MODALITIES,channel_names=names,signal_shape=list(signal.shape),
        source_hashes=hashes,identity_matches=int(data['subject'].item())==subject,
        sampling_note='Supplied synchronized row grid, 2000 rows/s; inertial acquisition rate is not inferred from row count.')
    json_write(directory/'identity.json',identity)
    if not identity['identity_matches']:
        print(f'Identity discrepancy for folder S{subject}: internal subject {identity["internal_subject"]}; retained and recorded.')
    return dict(signal=signal,metadata=metadata,stimulus=stimulus,channel_names=names)


In [ ]:
class ModalityWindows(Dataset):
    """All split boundaries and windows are independent of selected modalities."""
    def __init__(self, recording, split):
        self.recording=recording
        self.samples=Config.WIN_SAMPLES
        rows=[]
        for rep in recording['metadata'].to_dict('records'):
            if rep['split'] != split: continue
            for end in range(rep['run_start']+Config.TRIM_SAMPLES+self.samples,
                             rep['run_end']-Config.TRIM_SAMPLES+1,Config.STEP_SAMPLES):
                start=end-self.samples
                phase=((start+end)/2-rep['run_start'])/(rep['run_end']-rep['run_start'])
                rows.append(dict(**rep,window_start=start,window_end=end,
                    phase=['early','middle','late'][min(2,int(phase*3))],
                    stimulus_disagreement_fraction=float(np.mean(recording['stimulus'][start:end]!=rep['gesture']))))
        self.meta=pd.DataFrame(rows)
        assert len(self.meta) and self.meta.gesture.nunique()==17
        self.starts=self.meta.window_start.to_numpy(dtype=np.int64)
        self.y=self.meta.gesture.to_numpy(dtype=np.int64)-1
        channels=recording['signal'].shape[1]
        self.mean=np.zeros((channels,1),np.float32)
        self.std=np.ones((channels,1),np.float32)
    def __len__(self): return len(self.y)
    def physical(self,index):
        start=self.starts[index]
        return self.recording['signal'][start:start+self.samples].T.copy()
    def __getitem__(self,index):
        return torch.from_numpy((self.physical(index)-self.mean)/self.std), int(self.y[index])


In [ ]:
def train_scaler(dataset):
    total=np.zeros(dataset.mean.shape[0],np.float64); squares=total.copy(); count=0
    for begin in range(0,len(dataset),64):
        x=np.stack([dataset.physical(i) for i in range(begin,min(begin+64,len(dataset)))]).astype(np.float64)
        total+=x.sum(axis=(0,2)); squares+=(x*x).sum(axis=(0,2));count+=x.shape[0]*x.shape[2]
    mean=total/count;std=np.sqrt(np.maximum(squares/count-mean*mean,0))+1e-8
    return mean.astype(np.float32)[:,None],std.astype(np.float32)[:,None]


## 2. Freeze the earlier window identities
Each newly loaded split must match its stored SHA-256 window-key hash before training.


In [ ]:
# Saved window identities from the completed all-sensor study.
REFERENCE_WINDOW_HASHES = {1: {'train': '80b1fb7273a4fc4f38cf164385e73402cc9003f885b508ab1cf4d736994be9df', 'validation': 'beda8db0b7a21909ad85db0dafcb741f8a5ac1b7b01ddd01fad3c754535394f9', 'test': 'efef2ef1b2ac9852588472a8a9d3fcae804cd9f3bcc5df2b4b4caa5dd3bb94a4'}, 2: {'train': '5862bdbf12bd71626464eececab2e59b4d33c0d71857cfce5890620d281ae02d', 'validation': '4981b6d27f9e1c0733b5558037a83edd81ba259c97dcecb6343de0088d07ded3', 'test': '4a0f0f6058cd91e3187c1fd05a12e009671c49fa9779e1a203b730984b587274'}, 3: {'train': 'd6cb73ca654c168ddc7691570957b2236566c052eb5b6fdfb6ce9f21d5435681', 'validation': '090ecee107fa4399c53229de9999afbabe6a29e6d1ac9cf120c836fe586bb215', 'test': '1ed1c529ffb3b41c7c14e2abb8554e4f6dc9b782cb6142759ef4ab99a14c46f0'}, 4: {'train': 'c8bef73d6098c894af4becee8d0abcf5ffb0f46ef301038ee0a945c346574bbd', 'validation': '2f39a1f6b1bd6b17e96b7997be6c2a101690848cffbca6f4611f26d0c16eac7a', 'test': 'c16bd20a08544d3f2b14d5271c6ef96acdd9fb6d31ef094bcac977ecd56f256f'}, 5: {'train': '09cc8161076739fe1a3e4e5526dd93e523d34b1c6d6af8558606e49d53a27c36', 'validation': '0c5d45841bdb6b8d370f1d809cca0f147b9ec57cb16cf74ca17e5b3e230b009d', 'test': 'f727f553e0b6b8feec10963f01f0fd38fad48dedc1c4eabb9424b146af5cafff'}, 6: {'train': '4b9d1e0a5a7d5850f0ce8ac8e12cfa5f1e311b8155ef04ca7c4ae79bfb5f7ebb', 'validation': '1d80ec9fe5de7e42176f7abcb615157d57a1b41066506895e58574eb634cb60c', 'test': '984f6922385908e78b1b8e37f6f70ddca28348b9247953f384f2b0aa729d9d7c'}, 7: {'train': 'bf76d81905815cd9964f97288060b1a0dfb919f3a822ce21b2d1cbcbd1983742', 'validation': 'fb3e6c34dd12ed9e2a2531506d21d0a14cb193cf5c009b92dee47e6f2c74c978', 'test': 'ce68139cc2280f4c181b74ad0e30fa23addecb9b46aed53de83a44dab0298bf0'}, 8: {'train': 'b9e52252ddabc3b1af10e9f1f2d4cc29c0d913f0a57e83659f8029482d42f323', 'validation': '598f6bcf9c09d4d8975c070fad03b5d919375266fd0e672470d36085d39996b8', 'test': 'f907f45c0f7c543a0f21e62ef49808333dafb242abfc24b8e14f97aa5a3dcf7c'}, 9: {'train': '5183986e4c085f93d204185e368262301a334d15e1c7066746ab2b0f4a63bc07', 'validation': '2d50249868b3716552ab34746ce8eae885f7435841fc8fb81f505ee83c103b9b', 'test': '85e114db224b518fd876d9cbb979e1e55e072f3504ab8c7c67cfd60269aa557f'}, 10: {'train': 'fd2301da0c54467c1bc455bd36736b9611540111fb9a38f6e920160ed6abf2db', 'validation': 'e0a66d9ad00e79943511a86d7ca811cbfadbc9daa6d233dc7890339dfb71bb47', 'test': 'f52b8402f179784450420fb0c27664fd994c8e7583a5638259a2035d3b2be206'}, 11: {'train': '78bda9c12adb30da240621d0b57ce71a9e1bd8cf6ab8a7248791e92e9b506417', 'validation': 'e1dce03bfe311834677b4acdd4ee2c0bf09b3f473f84c3cd53e79ccc0dfcccbb', 'test': '83978a2026e6ba32f3915a4093340eb8a1f334c1c3396e7abfefbb900432bcb3'}, 12: {'train': 'c2f0d8a2ae46427a77301e316ecce34614df7f51363254f0d60e7825f262501c', 'validation': '89df3ca1774a93793889f572cc7da05b87e6872ae94334ff74a07eea813bc9d6', 'test': 'e0707eedff8327a21b16c83ccf8b7ead9fe8e2b6295c6bce0e1b88f428333036'}, 13: {'train': 'b4e55d35f7704642098f8fb8d37fe75184af6dcfff9fbb82c9f274c76df0a2a1', 'validation': '1fcc5b92170fe44d596ce65e2f8c7ead4fbfec6488e5380719214ea8408a7d58', 'test': '5113645e03c26247dfa84f19ca0e9a60e87b4ebfa631e38d3c6c99bcfd4b58e6'}, 14: {'train': 'a78980e1f8a94cf29988941ffe7baf4ed8976da30d81ad4a8c6018c6b7a88384', 'validation': '83c04ac4449e840d7124c2856742d07ec232be9aa46d3d5e34d517c55281695d', 'test': '187caf24e1b5da3f5930ed70c46ae5f02189d51ea3653ef948ec2dcd06697e2d'}, 15: {'train': 'd8f8cdb0de3d78b9ffd216e3156f2fb8441547cf4b2579c8805d9877e197dc29', 'validation': '083826e87923da9f25d51c6b655d3fb8f1a4360d85f664e9a0f6b784c2c4affa', 'test': '643e295b0ac318347e26e7cb2137b68361d3ade3d2567c70c84c2996642285fd'}, 16: {'train': 'bc7570a78674ed35aa636ac9955e9873f50ad718417e5a6530cb92c089cdeca4', 'validation': '44c9cd3a34b330b56d50445e64e2bf3c05520300c3bfd952946bebadf1f756e4', 'test': '6fc38926bf4434a3e2d593047b8281a27e7106fa92e5b153960caf845268a05c'}, 17: {'train': '09ddd5493081deda525bd5e4a6120697242e949cf91ca827fb1c93dfebb91d16', 'validation': '4f382dfb6d80a7df5498a0217aa54ddef33b79e65881c8e91ab198b2861d200f', 'test': 'eaa1990207e9997f6905be7bb607ddb0699c88e35a4362256174013b20117ab2'}, 18: {'train': '3d56463508c2f75fe3d6b6259f3a182be1a134e63d746e129a45788abf97262c', 'validation': 'bfb40eacf4de91f3b3fdc903845b7340f1c8d5c4e53fcfc4354e60203c685fa4', 'test': '6a386455b4962f917c13f664c23d1d131328d80ab737fb3037a2f311a4a33ac3'}, 19: {'train': '0e2fdfeb50c79e6855ec5a253e0d090b4ff773db6842e6715f269e33f5af878e', 'validation': '024b09c7a3cf68af5224bba6c573c0bff1096f2708d301b8a37061de2bb1fbfc', 'test': '8ab7f74a13148de0bac9cdab6706832e706be9d9892dec7783dc8cb639e5157d'}, 20: {'train': 'f6bf2745c817d0cf3680ac0bcb44c2b922fbbabf39d60c44be3ecf64bffa15ea', 'validation': '7bb323e56471df0a38ceac51e3898350ac15fe7f3265ddcaccfb3c11db43f2f3', 'test': '4811c69f8c9c22ca5c3845c456706d0a60d514a92f7d55f5eee16a05ee9ac09c'}, 21: {'train': 'e4b482bc1857e83f52aa471931ec77f250b35f8f40dd0dbed2447dd63aeeec72', 'validation': 'b6dd1fab0a4932424f43a8b6b0ae7d27ede6cd918b93ef9af792b2eeb880e76a', 'test': '1fef826c013e751e933b33372a175a54e8cb5e536b2a0bff50a75cd1e5683dc7'}, 22: {'train': '5fb1e46709a8ac0c3608931f531599986b3054957500679699a133588aa3ae73', 'validation': 'b670913276b71b3cba525e858f679a098e096ffbc30f7b69f58f24d3ddbc721e', 'test': 'd449fe2bb3de66a562b5672e11f57e27f60e295689b9a0537b28c7c40a9dc3bc'}}


## 3. Three-branch architecture
EMG waveform + log-power spectrogram + ACC/gyro/magnetometer; seven aligned frames; four ordered frequency regions; 551,542 parameters.


In [ ]:
import io as checkpoint_io


In [ ]:
from collections.abc import Iterable


In [ ]:
from typing import Any


In [ ]:
import torch


In [ ]:
from torch import Tensor, nn


In [ ]:
from torch.nn import functional as F


In [ ]:
EXPECTED_PARAMETERS = 551_542


In [ ]:
EXPECTED_PARAMETER_BREAKDOWN = {
    'waveform': 81_492,
    'spectral': 67_936,
    'inertial': 123_760,
    'fusion': 41_216,
    'temporal': 197_888,
    'attention': 4_161,
    'classifier': 35_089,
}


In [ ]:
FRAME_SAMPLES = 200


In [ ]:
FRAME_HOP = 100


In [ ]:
WINDOW_SAMPLES = 800


In [ ]:
N_FRAMES = 7


In [ ]:
def unfold_frames(signal: Tensor) -> Tensor:
    """Return [batch, channels, 7, 200], using only the supplied 800 rows."""
    if signal.ndim != 3 or signal.shape[-1] != WINDOW_SAMPLES:
        raise ValueError(f'Expected [batch, channels, 800], received {tuple(signal.shape)}')
    return signal.unfold(-1, FRAME_SAMPLES, FRAME_HOP)


In [ ]:
def log_power(z_emg: Tensor, hann: Tensor | None = None) -> Tensor:
    """Return unscaled log-power [batch, 12, 44, 7] for 20:10:450 Hz.

    ``z_emg`` has already received the fixed training input scaler. Explicit
    200-sample frames avoid the FFT-length-dependent framing of torch.stft.
    The FFT is at least float32, including inside an autocast context: CUDA
    half-precision FFTs cannot implement this non-power-of-two length.
    """
    if z_emg.ndim != 3 or z_emg.shape[1:] != (12, WINDOW_SAMPLES):
        raise ValueError(f'Expected [batch, 12, 800], received {tuple(z_emg.shape)}')
    if not z_emg.is_floating_point():
        raise TypeError('EMG input must be a floating-point tensor')
    fft_dtype = torch.float64 if z_emg.dtype == torch.float64 else torch.float32
    if hann is None:
        hann = torch.hann_window(FRAME_SAMPLES, periodic=True,
                                 device=z_emg.device, dtype=fft_dtype)
    else:
        hann = hann.to(device=z_emg.device, dtype=fft_dtype)
        if hann.shape != (FRAME_SAMPLES,):
            raise ValueError('Hann window must contain exactly 200 samples')
    with torch.autocast(device_type=z_emg.device.type, enabled=False):
        frames = unfold_frames(z_emg.to(dtype=fft_dtype))
        spectrum = torch.fft.rfft(frames * hann, n=FRAME_SAMPLES, dim=-1)
        power = spectrum.abs().square() / hann.square().sum()
        # [B, 12, frame, frequency] -> [B, 12, frequency, frame].
        return torch.log(power[..., 2:46] + 1e-8).permute(0, 1, 3, 2).contiguous()


In [ ]:
def _conv_bn_relu(in_channels: int, out_channels: int, kernel: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv1d(in_channels, out_channels, kernel, padding=kernel // 2, bias=False),
        nn.BatchNorm1d(out_channels, eps=1e-5, momentum=0.1),
        nn.ReLU(),
    )


In [ ]:
class ChannelSqueezeExcitation(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(channels, channels // 8), nn.ReLU(),
            nn.Linear(channels // 8, channels), nn.Sigmoid(),
        )

    def forward(self, x: Tensor) -> Tensor:
        return x * self.gate(x.mean(-1)).unsqueeze(-1)


In [ ]:
class FrameMultiKernelBlock(nn.Module):
    """Three single-convolution paths, plus projected residual and frame SE."""
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.paths = nn.ModuleList(_conv_bn_relu(in_channels, 32, k) for k in (3, 5, 7))
        self.merge = nn.Sequential(
            nn.Conv1d(96, out_channels, 1, bias=False),
            nn.BatchNorm1d(out_channels, eps=1e-5, momentum=0.1),
        )
        self.skip = nn.Conv1d(in_channels, out_channels, 1, bias=False)
        self.se = ChannelSqueezeExcitation(out_channels)

    def forward(self, x: Tensor) -> Tensor:
        merged = self.merge(torch.cat([path(x) for path in self.paths], dim=1))
        return self.se(F.relu(merged + self.skip(x)))


In [ ]:
class WaveformEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.stages = nn.Sequential(
            FrameMultiKernelBlock(12, 64), nn.MaxPool1d(2),
            FrameMultiKernelBlock(64, 96), nn.MaxPool1d(2),
        )
        self.projection = nn.Linear(192, 96)

    def forward(self, frames: Tensor) -> Tensor:
        # The same encoder processes every frame; batch and frame remain distinct.
        batch = frames.shape[0]
        x = frames.permute(0, 2, 1, 3).reshape(batch * N_FRAMES, 12, FRAME_SAMPLES)
        x = self.stages(x)
        x = F.relu(self.projection(torch.cat([x.mean(-1), x.amax(-1)], dim=1)))
        return x.reshape(batch, N_FRAMES, 96).transpose(1, 2).contiguous()


In [ ]:
class SpectralEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        for incoming, outgoing, kernel, stride in ((12, 32, 5, 2),
                                                   (32, 64, 5, 2),
                                                   (64, 96, 3, 1)):
            layers.extend([
                nn.Conv2d(incoming, outgoing, (kernel, 1), stride=(stride, 1),
                          padding=(kernel // 2, 0), bias=False),
                nn.BatchNorm2d(outgoing, eps=1e-5, momentum=0.1), nn.ReLU(),
            ])
        self.stages = nn.Sequential(*layers)
        self.projection = nn.Conv1d(384, 96, 1, bias=True)

    def forward(self, x: Tensor) -> Tensor:
        x = self.stages(x)
        # Preserve four ordered feature-frequency regions, rather than collapsing
        # absolute frequency location into a single global mean/max vector.
        regions = torch.stack([x[:, :, begin:end, :].mean(2)
                               for begin, end in ((0, 3), (3, 6), (6, 9), (9, 11))], dim=2)
        # Channel-major: each channel retains regions low -> high in adjacent slots.
        return F.relu(self.projection(regions.flatten(1, 2)))


In [ ]:
class InertialModalityEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.stages = nn.Sequential(
            _conv_bn_relu(36, 48, 5), nn.AvgPool1d(4),
            _conv_bn_relu(48, 64, 5),
        )
        self.projection = nn.Linear(128, 48)

    def forward(self, frames: Tensor) -> Tensor:
        x = self.stages(frames)
        return F.relu(self.projection(torch.cat([x.mean(-1), x.amax(-1)], dim=1)))


In [ ]:
class InertialEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.modalities = nn.ModuleList(InertialModalityEncoder() for _ in range(3))
        self.dynamic_projection = nn.Linear(144, 128)
        self.mean_projection = nn.Linear(108, 128)

    def forward(self, frames: Tensor) -> Tensor:
        batch = frames.shape[0]
        x = frames.permute(0, 2, 1, 3).reshape(batch * N_FRAMES, 108, FRAME_SAMPLES)
        encoded = [encoder(x[:, i * 36:(i + 1) * 36, :])
                   for i, encoder in enumerate(self.modalities)]
        dynamic = self.dynamic_projection(torch.cat(encoded, dim=1))
        static = self.mean_projection(x.mean(-1))
        # The mean is an additional feature; it is never subtracted from frames.
        output = F.relu(dynamic + static)
        return output.reshape(batch, N_FRAMES, 128).transpose(1, 2).contiguous()


In [ ]:
class ResidualTemporalBlock(nn.Module):
    def __init__(self, dilation: int, dropout: float) -> None:
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv1d(128, 192, 3, dilation=dilation, padding=dilation, bias=False),
            nn.BatchNorm1d(192, eps=1e-5, momentum=0.1), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(192, 128, 1, bias=False),
            nn.BatchNorm1d(128, eps=1e-5, momentum=0.1), nn.Dropout(dropout),
        )

    def forward(self, x: Tensor) -> Tensor:
        return F.relu(x + self.layers(x))


In [ ]:
class ThreeBranchC1(nn.Module):
    """Exact 551,542-parameter C1, compatible with Trainer's model(x) call."""
    def __init__(self, dropout: float = 0.15) -> None:
        super().__init__()
        self.waveform = WaveformEncoder()
        self.spectral = SpectralEncoder()
        self.inertial = InertialEncoder()
        self.fusion = nn.Sequential(
            nn.Conv1d(320, 128, 1, bias=False),
            nn.BatchNorm1d(128, eps=1e-5, momentum=0.1), nn.ReLU(),
        )
        self.temporal = nn.Sequential(ResidualTemporalBlock(1, dropout),
                                      ResidualTemporalBlock(2, dropout))
        self.attention = nn.Sequential(nn.Linear(128, 32), nn.Tanh(), nn.Linear(32, 1))
        self.classifier = nn.Sequential(nn.Linear(256, 128), nn.ReLU(),
                                        nn.Dropout(dropout), nn.Linear(128, 17))
        self.register_buffer('hann', torch.hann_window(FRAME_SAMPLES, periodic=True))
        self.register_buffer('spectral_mean', torch.zeros(12, 44))
        self.register_buffer('spectral_std', torch.ones(12, 44))
        self.register_buffer('spectral_constant', torch.zeros(12, 44, dtype=torch.bool))
        self.register_buffer('spectral_fitted', torch.tensor(False))
        self.register_buffer('spectral_fit_frames', torch.tensor(0, dtype=torch.long))
        if self.count_params() != EXPECTED_PARAMETERS:
            raise AssertionError(f'C1 parameter mismatch: {self.count_params()}')

    def count_params(self) -> int:
        return sum(parameter.numel() for parameter in self.parameters())

    def parameter_breakdown(self) -> dict[str, int]:
        return {name: sum(parameter.numel() for parameter in getattr(self, name).parameters())
                for name in EXPECTED_PARAMETER_BREAKDOWN}

    @torch.no_grad()
    def fit_spectral_scaler(self, training_batches: Iterable[Any], *, split: str = 'train',
                            std_floor: float = 1e-6) -> dict[str, Any]:
        """Fit frozen moments without running a CNN or updating BatchNorm.

        The iterable yields standardized x or (x,y) batches. One observation for
        each channel/frequency bin is one frame; all seven frames of every
        training window receive equal weight. Overlap is intentional. Population
        variance uses a stable float64 batched merge; SD below ``std_floor`` is
        flagged and clamped. A second fit is rejected to avoid accidental reuse
        on validation/test; create a fresh model for a new subject or fold.

        The explicit split guard cannot establish arbitrary generator provenance.
        The caller must save source/window hashes. A DataLoader exposing a dataset
        with a ``meta['split']`` column receives an additional provenance check.
        """
        if split != 'train':
            raise ValueError('Spectral scaler may be fitted only on the train split')
        if bool(self.spectral_fitted.item()):
            raise RuntimeError('Spectral scaler is already fitted; use a fresh model for another fold')
        if std_floor <= 0:
            raise ValueError('std_floor must be positive')
        dataset = getattr(training_batches, 'dataset', None)
        metadata = getattr(dataset, 'meta', None)
        if metadata is not None and 'split' in metadata:
            if set(metadata['split'].unique()) != {'train'}:
                raise ValueError('Spectral fitting DataLoader contains non-training metadata')
        device = self.spectral_mean.device
        count = 0
        mean = torch.zeros(12, 44, dtype=torch.float64, device=device)
        m2 = torch.zeros_like(mean)
        for batch in training_batches:
            x = batch[0] if isinstance(batch, (tuple, list)) else batch
            x = torch.as_tensor(x, device=device, dtype=self.spectral_mean.dtype)
            if x.ndim != 3 or x.shape[1:] != (120, WINDOW_SAMPLES) or x.shape[0] == 0:
                raise ValueError('Spectral scaler requires nonempty [B,120,800] training batches')
            if not bool(torch.isfinite(x).all().item()):
                raise ValueError('Nonfinite training input in spectral scaler fit')
            values = log_power(x[:, :12], self.hann).to(torch.float64)
            batch_count = values.shape[0] * N_FRAMES
            batch_mean = values.mean(dim=(0, 3))
            batch_m2 = (values - batch_mean[None, :, :, None]).square().sum(dim=(0, 3))
            combined_count = count + batch_count
            delta = batch_mean - mean
            m2 += batch_m2 + delta.square() * (count * batch_count / combined_count)
            mean += delta * (batch_count / combined_count)
            count = combined_count
        if not count:
            raise ValueError('Cannot fit the spectral scaler on an empty iterable')
        std = (m2 / count).clamp_min(0).sqrt()
        self.spectral_mean.copy_(mean)
        self.spectral_constant.copy_(std < std_floor)
        self.spectral_std.copy_(std.clamp_min(std_floor))
        self.spectral_fit_frames.fill_(count)
        self.spectral_fitted.fill_(True)
        return {
            'split': 'train', 'windows': count // N_FRAMES, 'frames': count,
            'statistic': 'float64_population_moments_over_training_windows_and_seven_frames',
            'std_floor': float(std_floor),
            'constant_bins': int(self.spectral_constant.sum().item()),
            'mean': self.spectral_mean.detach().cpu().tolist(),
            'std': self.spectral_std.detach().cpu().tolist(),
            'constant': self.spectral_constant.detach().cpu().tolist(),
        }

    def forward_features(self, x: Tensor, *, retain_sequences: bool = False) -> dict[str, Any]:
        if x.ndim != 3 or x.shape[1:] != (120, WINDOW_SAMPLES):
            raise ValueError(f'C1 expects [B,120,800], received {tuple(x.shape)}')
        if not bool(self.spectral_fitted.item()):
            raise RuntimeError('Fit the spectral scaler using training windows before model(x)')
        frames = unfold_frames(x)
        waveform = self.waveform(frames[:, :12])
        power = log_power(x[:, :12], self.hann)
        scaled_power = ((power - self.spectral_mean[None, :, :, None]) /
                        self.spectral_std[None, :, :, None])
        spectral = self.spectral(scaled_power)
        inertial = self.inertial(frames[:, 12:])
        fused = self.temporal(self.fusion(torch.cat([waveform, spectral, inertial], dim=1)))
        scores = self.attention(fused.transpose(1, 2)).squeeze(-1)
        attention = torch.softmax(scores, dim=-1)
        weighted = (fused * attention[:, None, :]).sum(-1)
        embedding = torch.cat([weighted, fused.mean(-1)], dim=1)
        result = {
            'logits': self.classifier(embedding), 'embedding': embedding,
            'branch_embeddings': {'waveform': waveform.mean(-1),
                                  'spectral': spectral.mean(-1),
                                  'inertial': inertial.mean(-1)},
            'attention': attention,
        }
        if retain_sequences:
            result['sequences'] = {'waveform': waveform, 'spectral': spectral,
                                   'inertial': inertial, 'fused': fused}
        return result

    def forward(self, x: Tensor) -> Tensor:
        return self.forward_features(x)['logits']


In [ ]:
def model_preflight(device: str | torch.device = 'cpu') -> dict[str, Any]:
    """Meaningful synthetic checks for the installed Kaggle PyTorch runtime.

    Does not touch recordings or disk, install packages, or start an experiment.
    CPU and applicable CUDA RNG state are restored before return.
    """
    target = torch.device(device)
    # torch.manual_seed also seeds CUDA generators. Preserve every initialized
    # device's stream instead of changing an unused second Kaggle GPU's RNG.
    cuda_devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []
    with torch.random.fork_rng(devices=cuda_devices):
        torch.manual_seed(941)
        model = ThreeBranchC1().to(target)
        assert model.parameter_breakdown() == EXPECTED_PARAMETER_BREAKDOWN
        x = torch.randn(3, 120, WINDOW_SAMPLES, device=target)
        try:
            model(x)
        except RuntimeError as error:
            assert 'spectral scaler' in str(error)
        else:
            raise AssertionError('Unfitted spectral scaling must block model inference')
        try:
            model.fit_spectral_scaler([x], split='test')
        except ValueError as error:
            assert 'train split' in str(error)
        else:
            raise AssertionError('The spectral scaler must reject non-training splits')
        frames = unfold_frames(x)
        assert frames.shape == (3, 120, 7, 200)
        for frame in range(N_FRAMES):
            torch.testing.assert_close(frames[:, :, frame], x[:, :, frame * 100:frame * 100 + 200])
        # Independent explicit-frame transform verifies frequency/frame ordering.
        hann = torch.hann_window(200, periodic=True, device=target)
        reference = torch.stack([
            torch.log(torch.fft.rfft(x[:, :12, begin:begin + 200] * hann, dim=-1)
                      .abs().square()[..., 2:46] / hann.square().sum() + 1e-8)
            for begin in range(0, 601, 100)], dim=-1)
        torch.testing.assert_close(log_power(x[:, :12]), reference)
        before_bn = {name: value.clone() for name, value in model.named_buffers()
                     if 'running_' in name or 'num_batches_tracked' in name}
        statistics = model.fit_spectral_scaler([(x[:2], torch.zeros(2)),
                                                (x[2:], torch.zeros(1))], split='train')
        assert statistics['windows'] == 3 and statistics['frames'] == 21
        try:
            model.fit_spectral_scaler([x], split='train')
        except RuntimeError as error:
            assert 'already fitted' in str(error)
        else:
            raise AssertionError('A fitted spectral scaler must reject accidental refitting')
        reference_double = reference.double()
        expected_mean = reference_double.mean(dim=(0, 3))
        expected_std = reference_double.permute(1, 2, 0, 3).reshape(12, 44, -1).std(-1, correction=0)
        torch.testing.assert_close(model.spectral_mean, expected_mean.float(), rtol=2e-5, atol=2e-6)
        torch.testing.assert_close(model.spectral_std, expected_std.float(), rtol=2e-5, atol=2e-6)
        for name, value in model.named_buffers():
            if name in before_bn:
                torch.testing.assert_close(value, before_bn[name], rtol=0, atol=0)
        frozen_scaler = {name: value.clone() for name, value in model.named_buffers()
                         if name.startswith('spectral_')}
        model.train()
        outputs = model.forward_features(x, retain_sequences=True)
        assert outputs['logits'].shape == (3, 17)
        assert outputs['embedding'].shape == (3, 256)
        assert outputs['attention'].shape == (3, 7)
        for branch, width in (('waveform', 96), ('spectral', 96), ('inertial', 128), ('fused', 128)):
            assert outputs['sequences'][branch].shape == (3, width, 7)
        torch.testing.assert_close(outputs['attention'].sum(-1), torch.ones(3, device=target))
        loss = F.cross_entropy(outputs['logits'], torch.tensor([0, 8, 16], device=target))
        loss.backward()
        for name, module in (('waveform', model.waveform), ('spectral', model.spectral),
                             ('acc', model.inertial.modalities[0]),
                             ('gyro', model.inertial.modalities[1]),
                             ('mag', model.inertial.modalities[2]),
                             ('inertial_mean', model.inertial.mean_projection),
                             ('fusion', model.fusion)):
            gradients = [p.grad for p in module.parameters() if p.grad is not None]
            assert gradients and all(bool(torch.isfinite(g).all().item()) for g in gradients), name
            assert any(bool(g.abs().sum().item() > 0) for g in gradients), name
        for name, value in model.named_buffers():
            if name in frozen_scaler:
                torch.testing.assert_close(value, frozen_scaler[name], rtol=0, atol=0)
        model.eval()
        with torch.no_grad():
            expected = model(x)
            # An input to another branch cannot change waveform/spectral tokens.
            changed = x.clone()
            changed[:, 12:] += 1.75
            original_features = model.forward_features(x, retain_sequences=True)
            altered_features = model.forward_features(changed, retain_sequences=True)
            for name in ('waveform', 'spectral'):
                torch.testing.assert_close(original_features['sequences'][name],
                                           altered_features['sequences'][name], rtol=0, atol=0)
            checkpoint = checkpoint_io.BytesIO()
            torch.save(model.state_dict(), checkpoint)
            checkpoint.seek(0)
            restored = ThreeBranchC1().to(target)
            restored.load_state_dict(torch.load(checkpoint, map_location=target, weights_only=True))
            restored.eval()
            torch.testing.assert_close(restored(x), expected, rtol=0, atol=0)
        return {'success': True, 'parameters': model.count_params(),
                'parameter_breakdown': model.parameter_breakdown(), 'device': str(target),
                'torch_version': torch.__version__, 'frames': 7, 'frequency_bins': 44,
                'output_shape': list(expected.shape),
                'checks': ['exact_parameter_count', 'frame_alignment', 'explicit_fft_reference',
                           'unfitted_nontraining_and_refit_guards',
                           'training_spectral_moments', 'scaler_fit_does_not_update_batchnorm',
                           'frozen_scaler', 'forward_backward_all_branches',
                           'branch_input_isolation', 'checkpoint_round_trip']}


In [ ]:
c1_preflight = model_preflight


## 4. Training, checkpoint evaluation and failure diagnostics
All scalers are frozen before validation/test evaluation. Selected-epoch training metrics use evaluation mode.


In [ ]:
import gc


In [ ]:
import hashlib


In [ ]:
import json


In [ ]:
import random


In [ ]:
import shutil


In [ ]:
import sys


In [ ]:
import time


In [ ]:
import traceback


In [ ]:
from datetime import datetime, timezone


In [ ]:
from pathlib import Path


In [ ]:
import numpy as np


In [ ]:
import pandas as pd


In [ ]:
import matplotlib.pyplot as plt


In [ ]:
import torch


In [ ]:
import torch.nn.functional as F


In [ ]:
from torch.utils.data import DataLoader


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis


In [ ]:
from sklearn.preprocessing import StandardScaler


In [ ]:
from sklearn.pipeline import make_pipeline


In [ ]:
WINDOW_KEYS = ['subject', 'gesture', 'native_repetition', 'window_start', 'window_end']


In [ ]:
ARMS = ('B0', 'C1')


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [ ]:
def make_model(arm):
    if arm == 'B0':
        return OriginalMultiKernelAttention1DCNN(120, 17, Config.DROPOUT)
    if arm == 'C1':
        return ThreeBranchC1(dropout=Config.DROPOUT)
    raise ValueError(arm)


In [ ]:
def window_hashes(datasets):
    return {split: hashlib.sha256(ds.meta[WINDOW_KEYS].to_csv(index=False).encode()).hexdigest()
            for split, ds in datasets.items()}


In [ ]:
def preflight_models(root):
    record = {'C1': c1_preflight(Config.DEVICE)}
    seed_everything(123)
    model = make_model('B0').to(Config.DEVICE)
    assert sum(p.numel() for p in model.parameters()) == 552966
    x = torch.randn(2, 120, 800, device=Config.DEVICE)
    logits = model(x)
    F.cross_entropy(logits, torch.tensor([0, 16], device=Config.DEVICE)).backward()
    assert logits.shape == (2, 17) and torch.isfinite(logits).all()
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
    record['B0'] = {'success': True, 'parameters': 552966}
    record['success'] = True
    json_write(root / 'preflight.json', record)
    del model, x, logits
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
def predict_arm(model, dataset, arm):
    if arm == 'B0':
        probability, embedding, attention = predict_with_diagnostics(model, dataset, retain_attention=True)
        return dict(probabilities=probability, embeddings=embedding, attention=attention,
                    branch_embeddings={})
    model.eval()
    probabilities, embeddings, attention = [], [], []
    branches = {'waveform': [], 'spectral': [], 'inertial': []}
    with torch.no_grad():
        for x, _ in DataLoader(dataset, batch_size=Config.BATCH_SIZE, shuffle=False):
            result = model.forward_features(x.to(Config.DEVICE))
            probabilities.append(result['logits'].softmax(1).cpu().numpy())
            embeddings.append(result['embedding'].cpu().numpy())
            attention.append(result['attention'].cpu().numpy())
            for name in branches:
                branches[name].append(result['branch_embeddings'][name].cpu().numpy())
    return dict(probabilities=np.concatenate(probabilities), embeddings=np.concatenate(embeddings),
                attention={'temporal': np.concatenate(attention)},
                branch_embeddings={k: np.concatenate(v) for k, v in branches.items()})


In [ ]:
def export_predictions(model, dataset, arm, subject, seed_base, split, directory):
    directory.mkdir(parents=True, exist_ok=True)
    tick = time.perf_counter()
    result = predict_arm(model, dataset, arm)
    elapsed = time.perf_counter() - tick
    p = result['probabilities']
    assert p.shape == (len(dataset), 17) and np.isfinite(p).all()
    assert np.allclose(p.sum(1), 1, atol=1e-5)
    frame = dataset.meta.copy()
    frame['arm'], frame['seed_base'] = arm, seed_base
    frame['prediction'], frame['confidence'] = p.argmax(1) + 1, p.max(1)
    frame['correct'] = frame['prediction'] == frame['gesture']
    frame['window_ms'], frame['stride_ms'] = 400, 100
    frame['endpoint_ms'] = (frame.window_end - frame.run_start) / 2
    frame['phase_fraction'] = ((frame.window_start + frame.window_end) / 2 - frame.run_start) / (frame.run_end - frame.run_start)
    frame.to_csv(directory / 'predictions.csv', index=False)
    np.savez_compressed(directory / 'probabilities_embeddings.npz', y_true=dataset.y,
                        probabilities=p, embeddings=result['embeddings'],
                        **{f'branch_{k}': v for k, v in result['branch_embeddings'].items()})
    if result['attention']:
        np.savez_compressed(directory / 'attention.npz', **result['attention'])
    metrics, calibration = probability_metrics(dataset.y, p)
    metrics.update(arm=arm, subject=subject, seed_base=seed_base, split=split,
                   evaluation_seconds=elapsed)
    json_write(directory / 'metrics.json', metrics)
    pd.DataFrame(calibration).to_csv(directory / 'calibration.csv', index=False)
    errors = frame.groupby(['gesture', 'native_repetition']).agg(windows=('correct', 'size'), correct=('correct', 'sum')).reset_index()
    errors['wrong'] = errors.windows - errors.correct
    errors['error_percent'] = 100 * errors.wrong / errors.windows
    errors.to_csv(directory / 'gesture_errors.csv', index=False)
    if split == 'test':
        fig, ax = plt.subplots(figsize=(11, 4))
        ax.bar(errors.gesture, errors.correct, label='Correct')
        ax.bar(errors.gesture, errors.wrong, bottom=errors.correct, label='Wrong')
        ax.set(title=f'S{subject:02} / seed {seed_base} / {arm}', xlabel='Exercise B gesture', ylabel='Test windows', xticks=range(1, 18))
        ax.legend()
        save_figure(fig, directory / 'gesture_errors.png')
    return metrics, result


In [ ]:
def export_input_features(dataset, subject_folder, seed_base):
    """Describe every test input once; predictions from either arm join by window key."""
    output = dataset.meta[WINDOW_KEYS + ['phase', 'stimulus_disagreement_fraction']].copy()
    output['seed_base'] = seed_base
    records = []
    for start in range(0, len(dataset), 64):
        stop = min(start + 64, len(dataset))
        physical = np.stack([dataset.physical(i) for i in range(start, stop)]).astype(np.float64)
        emg = physical[:, :12]
        rms25 = np.sqrt(np.mean(emg[:, :, -50:] ** 2, axis=-1))
        rms100 = np.sqrt(np.mean(emg[:, :, -200:] ** 2, axis=-1))
        rms400 = np.sqrt(np.mean(emg ** 2, axis=-1))
        mav = np.mean(np.abs(emg), axis=-1)
        spectrum = np.abs(np.fft.rfft(emg * np.hanning(800)[None, None], axis=-1)) ** 2
        frequencies = np.fft.rfftfreq(800, 1 / 2000)
        band = (frequencies >= 20) & (frequencies <= 450)
        powers = spectrum[:, :, band]
        mean_frequency = (powers * frequencies[band]).sum(-1) / np.maximum(powers.sum(-1), 1e-30)
        values = {}
        for c in range(12):
            for name, array in [('rms25', rms25), ('rms100', rms100), ('rms400', rms400), ('mav400', mav), ('mean_frequency400', mean_frequency)]:
                values[f'emg_{c+1:02}_{name}'] = array[:, c]
        for name, sl in [('acc', slice(12, 48)), ('gyro', slice(48, 84)), ('mag', slice(84, 120))]:
            x = physical[:, sl]
            mean, sd = x.mean(-1), x.std(-1)
            for c in range(36):
                values[f'{name}_{c+1:02}_mean'] = mean[:, c]
                values[f'{name}_{c+1:02}_std'] = sd[:, c]
        records.append(pd.DataFrame(values))
    pd.concat([output.reset_index(drop=True), pd.concat(records, ignore_index=True)], axis=1).to_csv(subject_folder / 'input_features.csv', index=False)


In [ ]:
def export_matched_cases(dataset, subject_folder, seed_base, predictions):
    """Save one failure and a same-gesture, nearby-phase success for each arm/gesture."""
    chosen = set()
    y = dataset.y
    phase = ((dataset.meta.window_start + dataset.meta.window_end) / 2 - dataset.meta.run_start) / (dataset.meta.run_end - dataset.meta.run_start)
    for arm, p in predictions.items():
        correct = p.argmax(1) == y
        for gesture in range(17):
            failed = np.flatnonzero((y == gesture) & ~correct)
            good = np.flatnonzero((y == gesture) & correct)
            if len(failed):
                index = int(failed[np.argmax(p[failed].max(1))])
                chosen.add(index)
                if len(good):
                    chosen.add(int(good[np.argmin(np.abs(phase.iloc[good].to_numpy() - phase.iloc[index]))]))
    if not chosen:
        return
    indices = np.asarray(sorted(chosen), dtype=np.int64)
    recording = dataset.recording
    # Load the original EMG only for selected cases; never retain whole raw IMU duplicates.
    from scipy.io import loadmat
    file_name = recording['metadata'].file_name.iloc[0]
    matches = list(Config.KAGGLE_INPUT.rglob(file_name))
    assert len(matches) == 1
    raw = loadmat(matches[0], variable_names=['emg', 'restimulus'])
    starts = dataset.starts[indices]
    np.savez_compressed(subject_folder / 'waveform_cases.npz',
        window_indices=indices, signal_filtered=np.stack([dataset.physical(i) for i in indices]),
        raw_emg=np.stack([raw['emg'][s:s+800].T for s in starts]),
        restimulus=np.stack([raw['restimulus'][s:s+800].reshape(-1) for s in starts]),
        stimulus=np.stack([recording['stimulus'][s:s+800] for s in starts]),
        **{f'probability_{arm}': p[indices] for arm, p in predictions.items()})
    cases = dataset.meta.iloc[indices].copy()
    cases['seed_base'] = seed_base
    cases['window_index'] = indices
    cases.to_csv(subject_folder / 'waveform_cases.csv', index=False)


In [ ]:
def fit_embedding_probes(results, datasets, directory):
    """Training-fitted diagnostic probes of frozen C1 embeddings, not new sensor CNNs."""
    rows = []
    for name in ('waveform', 'spectral', 'inertial', 'fused'):
        def representation(split):
            return results[split]['embeddings'] if name == 'fused' else results[split]['branch_embeddings'][name]
        probe = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'))
        probe.fit(representation('train'), datasets['train'].y)
        saved = {}
        for split, dataset in datasets.items():
            p = probe.predict_proba(representation(split))
            assert np.array_equal(probe.classes_, np.arange(17))
            metric, _ = probability_metrics(dataset.y, p)
            rows.append(dict(branch=name, split=split, **metric))
            saved[f'{split}_probabilities'] = p.astype(np.float32)
        np.savez_compressed(directory / f'probe_{name}.npz', **saved)
    pd.DataFrame(rows).to_csv(directory / 'embedding_probe_metrics.csv', index=False)


In [ ]:
def train_one(subject, seed_base, arm, datasets, hashes, folder):
    folder.mkdir(parents=True, exist_ok=True)
    actual_seed = seed_base + 1009 * subject
    seed_everything(actual_seed)
    model = make_model(arm).to(Config.DEVICE)
    initial_hash = hashlib.sha256(b''.join(p.detach().cpu().numpy().tobytes() for p in model.parameters())).hexdigest()
    manifest = dict(status='running', arm=arm, subject=subject, seed_base=seed_base, seed=actual_seed,
                    parameters=sum(p.numel() for p in model.parameters()), window_key_hashes=hashes,
                    initial_parameter_sha256=initial_hash,
                    **{f'{s}_windows': len(ds) for s, ds in datasets.items()})
    json_write(folder / 'fit_manifest.json', manifest)
    np.savez_compressed(folder / 'normalizer.npz', mean=datasets['train'].mean, std=datasets['train'].std,
                        constant_channels=datasets['train'].std[:, 0] <= 1.01e-8,
                        channel_names=datasets['train'].recording['channel_names'])
    spectral_seconds = 0.0
    if arm == 'C1':
        tick = time.perf_counter()
        stats = model.fit_spectral_scaler(DataLoader(datasets['train'], batch_size=64, shuffle=False), split='train')
        spectral_seconds = time.perf_counter() - tick
        json_write(folder / 'spectral_scaler.json', stats)
        np.savez_compressed(folder / 'spectral_scaler.npz', mean=model.spectral_mean.cpu().numpy(), std=model.spectral_std.cpu().numpy(),
                            constant=model.spectral_constant.cpu().numpy())
    train = DataLoader(datasets['train'], batch_size=128, shuffle=True, generator=torch.Generator().manual_seed(actual_seed))
    val = DataLoader(datasets['validation'], batch_size=128, shuffle=False)
    trainer = DiagnosticTrainer(model, folder / 'best_model.pt', 17)
    torch.cuda.reset_peak_memory_stats()
    trainer.fit(train, val)
    learning_report(trainer, folder)
    model.load_state_dict(torch.load(folder / 'best_model.pt', map_location=Config.DEVICE, weights_only=True))
    metrics, results = [], {}
    for split, dataset in datasets.items():
        metric, result = export_predictions(model, dataset, arm, subject, seed_base, split, folder / split)
        metrics.append(metric)
        results[split] = result
    pd.DataFrame(metrics).to_csv(folder / 'metrics.csv', index=False)
    json_write(folder / 'metrics.json', metrics)
    if arm == 'C1':
        fit_embedding_probes(results, datasets, folder)
    manifest.update(status='complete', selected_epoch=trainer.best_epoch, epochs_run=len(trainer.epoch_rows),
                    optimizer_steps=len(train) * len(trainer.epoch_rows), training_seconds=trainer.train_wall,
                    spectral_fit_seconds=spectral_seconds, peak_cuda_memory_bytes=torch.cuda.max_memory_allocated())
    json_write(folder / 'fit_manifest.json', manifest)
    json_write(folder / 'resource_metrics.json', {k: manifest[k] for k in ['parameters', 'training_seconds', 'spectral_fit_seconds', 'peak_cuda_memory_bytes']})
    test_probabilities = results['test']['probabilities']
    del model, trainer, train, val, results
    gc.collect()
    torch.cuda.empty_cache()
    return metrics, test_probabilities


In [ ]:
def run_three_branch_study():
    assert Config.DEVICE.type == 'cuda', 'Enable the Kaggle T4 GPU.'
    Config.SEED = 42  # Model seed changes must never change the repetition split.
    Config.MODALITIES = ('emg', 'acc', 'gyro', 'mag')
    Config.AUGMENT_TRAIN = Config.REFIT_ON_TRAIN_PLUS_VAL = False
    Config.EVALUATE_TEST = True
    seed_base = int(getattr(Config, 'SEED_BASE', 42))
    assert seed_base in (42, 43, 44)
    smoke = bool(getattr(Config, 'SMOKE', False))
    subjects = [1, 22] if smoke else list(range(1, 23))
    Config.MIN_EPOCHS, Config.MAX_EPOCHS, Config.PATIENCE = (1, 2, 2) if smoke else (20, 150, 15)
    Config.WIN_MS, Config.STEP_MS, Config.TRIM_MS = 400, 100, 100
    Config.WIN_SAMPLES, Config.STEP_SAMPLES, Config.TRIM_SAMPLES = 800, 200, 200
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    root = Config.KAGGLE_WORKING / f'db7_three_branch_{seed_base}_{stamp}'
    root.mkdir(parents=True)
    manifest = dict(arms=list(ARMS), seed_base=seed_base, split_seed=42, subjects=subjects, smoke=smoke,
                    automation=getattr(Config, 'AUTOMATION', {}), labels=list(range(1, 18)), modalities=list(Config.MODALITIES),
                    window_ms=400, stride_ms=100, trim_ms=100, internal_frame_ms=100, internal_hop_ms=50,
                    protocol='within-subject 4 train / 1 validation / 1 test repetitions',
                    checkpoint='minimum validation loss; no refit', acc_centering=False, gating=False,
                    augmentation=False, test_interpretation='previously inspected exploratory test split',
                    versions=dict(torch=torch.__version__, numpy=np.__version__, python=sys.version),
                    expected_parameters={'B0': 552966, 'C1': 551542})
    json_write(root / 'run_manifest.json', manifest)
    metrics, completed = [], []
    try:
        preflight_models(root)
        for subject in subjects:
            subject_folder = root / f'S{subject:02}' / f'seed_{seed_base}'
            subject_folder.mkdir(parents=True)
            recording = load_modalities(subject, subject_folder)
            datasets = {split: ModalityWindows(recording, split) for split in ('train', 'validation', 'test')}
            hashes = window_hashes(datasets)
            assert hashes == REFERENCE_WINDOW_HASHES[subject], f'S{subject}: historical window identities changed'
            mean, std = train_scaler(datasets['train'])
            assert np.isfinite(mean).all() and np.isfinite(std).all() and (std > 0).all()
            for split, ds in datasets.items():
                ds.mean, ds.std = mean, std
                ds.meta.to_csv(subject_folder / f'{split}_window_manifest.csv', index=False)
            export_input_features(datasets['test'], subject_folder, seed_base)
            predictions = {}
            for arm in ARMS:
                print(f'\nS{subject:02} / seed {seed_base} / {arm}', flush=True)
                rows, p = train_one(subject, seed_base, arm, datasets, hashes, subject_folder / arm)
                metrics.extend(rows)
                predictions[arm] = p
                completed.append(dict(subject=subject, seed_base=seed_base, arm=arm))
                pd.DataFrame(metrics).to_csv(root / 'all_subject_metrics.csv', index=False)
                json_write(root / 'progress.json', dict(completed_fits=completed))
            export_matched_cases(datasets['test'], subject_folder, seed_base, predictions)
            del datasets, recording, predictions, ds
            gc.collect()
            torch.cuda.empty_cache()
        pd.DataFrame(metrics).groupby(['arm', 'split'])[['accuracy', 'f1_macro']].mean().to_csv(root / 'mean_subject_metrics.csv')
        json_write(root / 'completion.json', dict(success=True, cnn_fits=len(completed), subjects=subjects,
                                                seed_base=seed_base, arms=list(ARMS)))
    except Exception:
        (root / 'FAILURE.txt').write_text(traceback.format_exc(), encoding='utf-8')
        raise
    finally:
        archive = shutil.make_archive(str(root), 'zip', root)
        print('Outputs:', root, '\nArchive:', archive, flush=True)
    return root


## 5. Run configuration
Set SMOKE=True for S1 and S22, two epochs each. SEED_BASE changes model initialization/shuffling only; split seed stays 42.


In [ ]:
Config.SEED_BASE = 42
Config.SMOKE = False
Config.AUTOMATION = {}


## 6. Train and save outputs
GitHub replaces this final cell with run-specific provenance and seed. Full training begins only after its separate smoke job succeeds.


In [ ]:
RESULTS_DIRECTORY = run_three_branch_study()
print(RESULTS_DIRECTORY)
